# Abraham (2020)

#### Libraries

In [41]:
from importlib import import_module
from tabulate import tabulate

from src import init, metrics
from src.algorithms import abraham2020

#### Parameters

In [42]:
DATASET_NAME = 'motor'           # ['adult', 'compas', 'crime', 'motor']
N_CLUSTERS = 4
RANDOM_STATE = 4
INIT_METHOD = 'kmeans_plusplus'

#### Load dataset

In [43]:
dataset_module = import_module(f"src.datasets.{DATASET_NAME}")
_, X, s, n_nonsensitive = dataset_module.load()

Loading the processed Motor Insurance dataset (motor) from 'data/datasets/motor/motor.csv'
╭───────────┬─────────────┬──────────────┬─────────────────────────────┬───────────────────────╮
│   dataset │   # objects │   # features │   # nonsensitive attributes │   sensitive attribute │
├───────────┼─────────────┼──────────────┼─────────────────────────────┼───────────────────────┤
│     motor │       36311 │          521 │                          11 │                   Age │
╰───────────┴─────────────┴──────────────┴─────────────────────────────┴───────────────────────╯


#### Load initialised centroids

In [44]:
init_centroids = init.load(
    dataset_name=DATASET_NAME, n_clusters=N_CLUSTERS,
    init_method=INIT_METHOD, random_state=RANDOM_STATE
    )

Loading centroids from 'data/init/motor/k=4/kmeans_plusplus/motor.k4.kmeans_plusplus.r4.csv'
  shape: (4, 521)


#### Run Abraham (2020)'s algorithm

In [45]:
c, centroids, info, stats = abraham2020.run(
    X=X.copy(), n_nonsensitive=n_nonsensitive, s=s.copy(),
    n_clusters=N_CLUSTERS, init_centroids=init_centroids,
    dataset_name=DATASET_NAME, init_method=INIT_METHOD,
    random_state=RANDOM_STATE
    )

Initiating Abraham (2020)'s algorithm
╭───────────┬───────────────────────┬──────────────┬─────────────────┬────────────────┬─────────────┬───────────┬────────────╮
│   dataset │   sensitive_attribute │   n_clusters │     init_method │   random_state │   algorithm │   lambda_ │   max_iter │
├───────────┼───────────────────────┼──────────────┼─────────────────┼────────────────┼─────────────┼───────────┼────────────┤
│     motor │                   Age │            4 │ kmeans_plusplus │              4 │ Abraham2020 │         1 │        100 │
╰───────────┴───────────────────────┴──────────────┴─────────────────┴────────────────┴─────────────┴───────────┴────────────╯
Running algorithm
╭────────┬────────────────┬─────────────────┬─────────────┬─────────────────┬──────────────────┬───────────────────┬──────────────────╮
│   iter │   utility loss │   fairness loss │   objective │   reassignments │   centroids time │   assignment time │   iteration time │
├────────┼────────────────┼──────────

In [46]:
import pandas as pd
with pd.ExcelWriter('./result/motor/Abraham_K4_4.xlsx') as writer:
    c.to_excel(writer, sheet_name='Cluster', index=False)
    centroids.to_excel(writer, sheet_name='Centroid', index=False)
    info.to_excel(writer, sheet_name='Info', index=False)
    stats.to_excel(writer, sheet_name='Stats', index=False)

#### Evaluate

In [47]:
scores = metrics.evaluate(
    X=X, n_nonsensitive=n_nonsensitive, s=s, c=c, centroids=centroids,
    n_clusters=N_CLUSTERS, window_size=3
    )
print(tabulate(scores.to_frame(), tablefmt='rounded_outline'))

╭─────────────────────────────────────┬─────────────╮
│ k-means objective                   │ 0.450666    │
│ max ks statistic                    │ 0.0733476   │
│ max emd                             │ 0.0322719   │
│ pooling window loss (size=3)        │ 0.130181    │
│ Modified Abraham (2020)'s deviation │ 3.78685e-07 │
│ Ziko (2021)'s fairness error        │ 0.975209    │
│ Bera (2019)'s generalised balance   │ 0.0596704   │
╰─────────────────────────────────────┴─────────────╯


In [48]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

# Plot the overall dataset distribution (dashed black)
sns.kdeplot(data=s, fill=False, color="black", linestyle="--", label="dataset")  # data1 as in data 2 we dont have age column. Also it will not affect as it drawing for whole dataset

# Plot each cluster's age distribution
clusters = sorted(c.unique())
palette = sns.color_palette("tab10", len(clusters))

for i, j in enumerate(clusters):
    subset = s[c == j]  # here for which points we will need original dataset or data1
    sns.kdeplot(subset, fill=False, color=palette[i], label=f"cluster_{i} ({len(subset)} objects)")

plt.xlabel("age")
plt.ylabel("proportion")
plt.title("Age Distribution : Clusters vs Full Dataset")
plt.legend()
plt.tight_layout()
#plt.show()
plt.savefig("./result/motor/Plot_Abraham_K4_4.png")
plt.close()